# MMLU Case Study Extraction

Extract compact case-study candidates from the MMLU solver-vs-judge K-factor comparison.

Categories:

- hard to solve + hard to judge
- easy to solve + hard to judge
- hard to solve + easy to judge

Also extracts a focused set for solver difficulty bin 5, which has unusually high mean judge score in the Part 1 chart.

In [ ]:
from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 220)


In [ ]:
K = 2
TOP_N = 10
HIGH_PCT = 75
LOW_PCT = 25
FOCUS_BIN = 5

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "K-Factor").exists():
    REPO_ROOT = next(p for p in Path.cwd().parents if (p / "K-Factor").exists())

comparison_dir = REPO_ROOT / "K-Factor" / "results" / "mmlu_solver_judge_comparison"
paired_path = comparison_dir / f"mmlu_k{K}_paired_items.csv"
percentile_path = comparison_dir / f"mmlu_k{K}_solver_judge_difficulty_percentiles.csv"
bin_summary_path = comparison_dir / f"mmlu_k{K}_difficulty_bin_summary.csv"
out_dir = REPO_ROOT / "K-Factor" / "paper_artifacts" / "mmlu" / "case_studies"
out_dir.mkdir(parents=True, exist_ok=True)

for path in [paired_path, percentile_path, bin_summary_path, out_dir]:
    print(path, path.exists())


In [ ]:
def parse_score_dict(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, dict) else {}
    except (ValueError, SyntaxError):
        return {}


def mean_dict_score(value):
    scores = parse_score_dict(value)
    vals = [float(v) for v in scores.values() if v is not None and not pd.isna(v)]
    return float(np.mean(vals)) if vals else np.nan


def sd_dict_score(value):
    scores = parse_score_dict(value)
    vals = [float(v) for v in scores.values() if v is not None and not pd.isna(v)]
    return float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan


paired = pd.read_csv(paired_path)
percentiles = pd.read_csv(percentile_path)[[
    "solver_item_id",
    "solver_difficulty_percentile",
    "judge_difficulty_percentile",
]]

items = paired.merge(percentiles, on="solver_item_id", how="left", validate="one_to_one")
items["solver_score_mean"] = items["solver_scores"].apply(mean_dict_score)
items["judge_score_mean"] = items["judge_scores"].apply(mean_dict_score)
items["solver_score_sd"] = items["solver_scores"].apply(sd_dict_score)
items["judge_score_sd"] = items["judge_scores"].apply(sd_dict_score)
items["difficulty_bin"] = pd.qcut(
    items["solver_difficulty_centered"],
    q=10,
    labels=False,
    duplicates="drop",
) + 1

bin_summary = pd.read_csv(bin_summary_path)

print(f"items: {items.shape}")
print(f"missing solver percentiles: {items['solver_difficulty_percentile'].isna().sum()}")
print(f"missing judge percentiles:  {items['judge_difficulty_percentile'].isna().sum()}")

display(bin_summary)
display(items[[
    "solver_item_id",
    "solver_source",
    "difficulty_bin",
    "solver_difficulty_centered",
    "solver_difficulty_percentile",
    "judge_difficulty_centered",
    "judge_difficulty_percentile",
    "solver_score_mean",
    "judge_score_mean",
]].head())


In [ ]:
def select_case_studies(df, category, mask, score_fn, top_n=TOP_N):
    selected = df.loc[mask].copy()
    selected["case_category"] = category
    selected["case_rank_score"] = score_fn(selected)
    return selected.sort_values("case_rank_score", ascending=False).head(top_n)


hard_solve_hard_judge = select_case_studies(
    items,
    "hard_to_solve__hard_to_judge",
    (items["solver_difficulty_percentile"] >= HIGH_PCT) & (items["judge_difficulty_percentile"] >= HIGH_PCT),
    lambda d: np.minimum(d["solver_difficulty_percentile"], d["judge_difficulty_percentile"]),
)

easy_solve_hard_judge = select_case_studies(
    items,
    "easy_to_solve__hard_to_judge",
    (items["solver_difficulty_percentile"] <= LOW_PCT) & (items["judge_difficulty_percentile"] >= HIGH_PCT),
    lambda d: np.minimum(100 - d["solver_difficulty_percentile"], d["judge_difficulty_percentile"]),
)

hard_solve_easy_judge = select_case_studies(
    items,
    "hard_to_solve__easy_to_judge",
    (items["solver_difficulty_percentile"] >= HIGH_PCT) & (items["judge_difficulty_percentile"] <= LOW_PCT),
    lambda d: np.minimum(d["solver_difficulty_percentile"], 100 - d["judge_difficulty_percentile"]),
)

case_studies = pd.concat(
    [hard_solve_hard_judge, easy_solve_hard_judge, hard_solve_easy_judge],
    ignore_index=True,
)

summary = pd.DataFrame([
    {
        "case_category": "hard_to_solve__hard_to_judge",
        "n_available": int(((items["solver_difficulty_percentile"] >= HIGH_PCT) & (items["judge_difficulty_percentile"] >= HIGH_PCT)).sum()),
        "n_selected": len(hard_solve_hard_judge),
    },
    {
        "case_category": "easy_to_solve__hard_to_judge",
        "n_available": int(((items["solver_difficulty_percentile"] <= LOW_PCT) & (items["judge_difficulty_percentile"] >= HIGH_PCT)).sum()),
        "n_selected": len(easy_solve_hard_judge),
    },
    {
        "case_category": "hard_to_solve__easy_to_judge",
        "n_available": int(((items["solver_difficulty_percentile"] >= HIGH_PCT) & (items["judge_difficulty_percentile"] <= LOW_PCT)).sum()),
        "n_selected": len(hard_solve_easy_judge),
    },
])

display(summary)
display(case_studies[[
    "case_category",
    "case_rank_score",
    "solver_item_id",
    "solver_source",
    "difficulty_bin",
    "solver_difficulty_percentile",
    "judge_difficulty_percentile",
    "solver_difficulty_centered",
    "judge_difficulty_centered",
    "solver_score_mean",
    "judge_score_mean",
]].sort_values(["case_category", "case_rank_score"], ascending=[True, False]))


In [ ]:
bin5_items = items.loc[items["difficulty_bin"] == FOCUS_BIN].copy()

bin5_summary = pd.DataFrame([
    {
        "difficulty_bin": int(FOCUS_BIN),
        "n_items": len(bin5_items),
        "solver_difficulty_mean": bin5_items["solver_difficulty_centered"].mean(),
        "judge_score_mean": bin5_items["judge_score_mean"].mean(),
        "judge_score_sd_across_items": bin5_items["judge_score_mean"].std(),
        "judge_difficulty_mean": bin5_items["judge_difficulty_centered"].mean(),
        "judge_difficulty_sd_across_items": bin5_items["judge_difficulty_centered"].std(),
    }
])

source_by_bin = (
    items
    .groupby(["difficulty_bin", "solver_source"], observed=True)
    .size()
    .reset_index(name="n_items")
    .sort_values(["difficulty_bin", "n_items"], ascending=[True, False])
)

bin5_source_counts = source_by_bin[source_by_bin["difficulty_bin"] == FOCUS_BIN].copy()

bin5_high_judge_score = bin5_items.sort_values("judge_score_mean", ascending=False).head(TOP_N)
bin5_low_judge_difficulty = bin5_items.sort_values("judge_difficulty_centered", ascending=True).head(TOP_N)

print("Bin 5 summary")
display(bin5_summary)
print("Bin 5 source counts")
display(bin5_source_counts)
print("Bin 5 highest judge-score examples")
display(bin5_high_judge_score[[
    "solver_item_id",
    "solver_source",
    "solver_difficulty_centered",
    "judge_difficulty_centered",
    "solver_score_mean",
    "judge_score_mean",
    "solver_prompt",
]].head())
print("Bin 5 lowest judge-difficulty examples")
display(bin5_low_judge_difficulty[[
    "solver_item_id",
    "solver_source",
    "solver_difficulty_centered",
    "judge_difficulty_centered",
    "solver_score_mean",
    "judge_score_mean",
    "solver_prompt",
]].head())


In [ ]:
case_columns = [
    "case_category",
    "case_rank_score",
    "solver_item_id",
    "solver_pair_id",
    "solver_source",
    "solver_gold_letter",
    "difficulty_bin",
    "solver_difficulty_centered",
    "solver_difficulty_centered_laplace_se",
    "solver_difficulty_percentile",
    "judge_difficulty_centered",
    "judge_difficulty_centered_laplace_se",
    "judge_difficulty_percentile",
    "solver_score_mean",
    "judge_score_mean",
    "solver_score_sd",
    "judge_score_sd",
    "solver_prompt",
    "solver_scores",
    "judge_scores",
]
compact_columns = [
    "case_category",
    "case_rank_score",
    "solver_item_id",
    "solver_pair_id",
    "solver_source",
    "solver_gold_letter",
    "difficulty_bin",
    "solver_difficulty_centered",
    "solver_difficulty_centered_laplace_se",
    "solver_difficulty_percentile",
    "judge_difficulty_centered",
    "judge_difficulty_centered_laplace_se",
    "judge_difficulty_percentile",
    "solver_score_mean",
    "judge_score_mean",
    "solver_score_sd",
    "judge_score_sd",
]

case_studies_full = case_studies[case_columns].copy()
case_studies_compact = case_studies[compact_columns].copy()

case_studies_full.to_csv(out_dir / f"mmlu_k{K}_case_studies_full.csv", index=False)
case_studies_full.to_json(out_dir / f"mmlu_k{K}_case_studies_full.json", orient="records", indent=2, force_ascii=False)
case_studies_compact.to_csv(out_dir / f"mmlu_k{K}_case_studies_compact.csv", index=False)
case_studies_compact.to_json(out_dir / f"mmlu_k{K}_case_studies_compact.json", orient="records", indent=2, force_ascii=False)
summary.to_csv(out_dir / f"mmlu_k{K}_case_study_summary.csv", index=False)
summary.to_json(out_dir / f"mmlu_k{K}_case_study_summary.json", orient="records", indent=2, force_ascii=False)

bin5_items.to_csv(out_dir / f"mmlu_k{K}_solver_bin{FOCUS_BIN}_items.csv", index=False)
bin5_summary.to_csv(out_dir / f"mmlu_k{K}_solver_bin{FOCUS_BIN}_summary.csv", index=False)
bin5_source_counts.to_csv(out_dir / f"mmlu_k{K}_solver_bin{FOCUS_BIN}_source_counts.csv", index=False)
bin5_high_judge_score[case_columns[2:]].to_csv(out_dir / f"mmlu_k{K}_solver_bin{FOCUS_BIN}_high_judge_score_examples.csv", index=False)
bin5_low_judge_difficulty[case_columns[2:]].to_csv(out_dir / f"mmlu_k{K}_solver_bin{FOCUS_BIN}_low_judge_difficulty_examples.csv", index=False)

print("Saved:")
for path in sorted(out_dir.glob(f"mmlu_k{K}_*.csv")):
    print(path.relative_to(REPO_ROOT))


In [ ]:
# Optional: inspect one category at a time without opening the full text-heavy file.
category = "easy_to_solve__hard_to_judge"
display(
    case_studies_compact
    .loc[case_studies_compact["case_category"] == category]
    .sort_values("case_rank_score", ascending=False)
)
